In [1]:
import os 
import random
import shutil
import rasterio

In [2]:
mask_list = os.listdir("../data/mango_dataset/MF_huggingface/train/masks")
img_list = os.listdir("../data/mango_dataset/MF_huggingface/train/images")

In [18]:
len(mask_list), len(img_list)

(34272, 34272)

In [23]:
mask_list[:5]

['region_mid_pos1000_masks.tif',
 'region_mid_pos1001_masks.tif',
 'region_mid_pos1002_masks.tif',
 'region_mid_pos1003_masks.tif',
 'region_mid_pos1004_masks.tif']

In [ ]:
img_list[:5]|

['region_mid_pos1.tif',
 'region_mid_pos10.tif',
 'region_mid_pos1000.tif',
 'region_mid_pos1001.tif',
 'region_mid_pos1002.tif']

In [3]:
c = 0
weak_count = 0
neg_count = 0
mid_count = 0
pos_count = 0
weak_list = []
neg_list = []
mid_list = []
pos_list = []

for i in range(len(img_list)):
    if "weak_pos" in img_list[i]: 
        weak_count += 1
        weak_list.append(img_list[i])
    if "neg" in img_list[i]:
        neg_count += 1
        neg_list.append(img_list[i])
    if "mid_pos" in img_list[i]:
        mid_count += 1
        mid_list.append(img_list[i])
    if "pos" in img_list[i]:
        pos_count += 1
        pos_list.append(img_list[i])

random.seed(2404)
random.shuffle(weak_list)
random.shuffle(neg_list)
random.shuffle(mid_list)
random.shuffle(pos_list)

In [139]:
f"{neg_list[i].split('.')[0]}_masks.tif"

'region_neg14571_masks.tif'

In [ ]:
for l in [weak_list, neg_list, mid_list, pos_list]:
    for i in range(len(l)):
        shutil.copy(f"../data/mango_dataset/MF_huggingface/train/images/{l[i]}", f"../data/mango_dataset/MF_huggingface/train/images/{l[i].split('.')[0]}_images.tif")
        shutil.copy(f"../data/mango_dataset/MF_huggingface/train/masks/{l[i].split('.')[0]}_masks.tif", f"../data/mango_dataset/MF_huggingface/train/masks/{l[i].split('.')[0]}_masks.tif")

In [23]:
count = 0
subset_list = []
i = 0

for l in [weak_list, neg_list, mid_list, pos_list]:
    count = 0
    i = 0
    while count < 230:
        with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/images", l[i])) as src:
            image = src.read()          # Shape: (bands, height, width)
        with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/masks", f"{l[i].split('.')[0]}_masks.tif")) as src:
            mask = src.read()

        if image.shape[1] == 256 and image.shape[2] == 256:
            count += 1
            subset_list.append(l[i])

            img_output_path = os.path.join("../data/mango_dataset/mango_binary_north/images", l[i])
            with rasterio.open(
                img_output_path,
                "w",
                driver="GTiff",
                height=image.shape[1],
                width=image.shape[2],
                count=image.shape[0],
                dtype=image.dtype,
            ) as dst:
                dst.write(image)
            
            # mask[mask == 0] = 0
            mask[mask == 255] = 1
            mask_output_path = os.path.join("../data/mango_dataset/mango_binary_north/masks", f"{l[i].split('.')[0]}_masks.tif")
            with rasterio.open(
                mask_output_path,
                "w",
                driver="GTiff",
                height=mask.shape[1],
                width=mask.shape[2],
                count=mask.shape[0],
                dtype=mask.dtype,
            ) as dst:
                dst.write(mask)

        i += 1

c:\Users\ASUS\AppData\Local\Programs\Python\Python38\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


In [21]:
os.path.isfile(os.path.join("../data/mango_dataset/MF_huggingface/train/images", l[i]))

True

In [22]:
l[i]

'region_weak_pos1573.tif'

In [176]:
weak_count = 0
i = 0

while weak_count < 90:
    with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/images", weak_list[i])) as src:
        image = src.read()          # Shape: (bands, height, width)
    with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/masks", f"{weak_list[i].split('.')[0]}_masks.tif")) as src:
        mask = src.read()

    if image.shape[1] == 256 and image.shape[2] == 256:
        weak_count += 1
        subset_list.append(weak_list[i])

        img_output_path = os.path.join("../data/mango_dataset/mango_subset/images", weak_list[i])
        with rasterio.open(
            img_output_path,
            "w",
            driver="GTiff",
            height=image.shape[1],
            width=image.shape[2],
            count=image.shape[0],
            dtype=image.dtype,
        ) as dst:
            dst.write(image)
        
        mask[mask == 0] = 7
        mask[mask == 255] = 3
        mask_output_path = os.path.join("../data/mango_dataset/mango_subset/masks", f"{weak_list[i].split('.')[0]}_masks.tif")
        with rasterio.open(
            mask_output_path,
            "w",
            driver="GTiff",
            height=mask.shape[1],
            width=mask.shape[2],
            count=mask.shape[0],
            dtype=mask.dtype,
        ) as dst:
            dst.write(mask)

    i += 1

In [158]:
np.unique(mask, return_counts=True)

(array([  0, 255], dtype=uint8), array([65266,   270], dtype=int64))

In [161]:
mask[mask == 0] = 7
mask[mask == 255] = 3

np.unique(mask, return_counts=True)

(array([3, 7], dtype=uint8), array([  270, 65266], dtype=int64))

In [166]:
with rasterio.open(mask_output_path) as src:
    image = src.read()    # Shape: (bands, height, width)
    profile = src.profile
    transform = src.transform
    crs = src.crs
print(image.shape)

(1, 256, 256)


c:\Users\ASUS\AppData\Local\Programs\Python\Python38\lib\site-packages\rasterio\__init__.py:333: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [168]:
np.unique(image, return_counts=True)

(array([3, 7], dtype=uint8), array([  270, 65266], dtype=int64))

In [150]:
profile

{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': None, 'width': 256, 'height': 256, 'count': 13, 'crs': None, 'transform': Affine(1.0, 0.0, 0.0,
       0.0, 1.0, 0.0), 'blockysize': 1, 'tiled': False, 'interleave': 'pixel'}

In [71]:
subset_list = neg_list[:140] + weak_list[:90]

In [91]:
weak_list[:5]

['region_weak_pos350.tif',
 'region_weak_pos1650.tif',
 'region_weak_pos4217.tif',
 'region_weak_pos4119.tif',
 'region_weak_pos526.tif']

In [151]:
with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/images", "region_weak_pos350.tif")) as src:
    image = src.read()          # Shape: (bands, height, width)
    profile = src.profile
    transform = src.transform
    crs = src.crs

print(image.shape)
profile

(13, 262, 262)


{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 262, 'height': 262, 'count': 13, 'crs': CRS.from_epsg(32617), 'transform': Affine(10.0, 0.0, 779650.0,
       0.0, -10.0, 2950830.0), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'deflate', 'interleave': 'pixel'}

In [145]:
profile

{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 262, 'height': 262, 'count': 13, 'crs': CRS.from_epsg(32617), 'transform': Affine(10.0, 0.0, 779650.0,
       0.0, -10.0, 2950830.0), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'deflate', 'interleave': 'pixel'}

In [111]:
import rasterio
path = '../Mangrove CaMau data/Mask_CaMau.tif'

with rasterio.open(path) as src:
    img = src.read()

print(img.shape)

(1, 5514, 12628)


In [112]:
np.unique(img)

array([1, 2, 3, 4, 5, 6], dtype=uint8)

In [117]:
img[0, :5, :5]

array([[6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6],
       [6, 6, 6, 6, 6]], dtype=uint8)

In [109]:
img[:, 1, :10]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=uint8)

In [102]:
with rasterio.open(os.path.join("../data/mango_dataset/MF_huggingface/train/masks", "region_weak_pos350_masks.tif")) as src:
    mask = src.read()          # Shape: (bands, height, width)
    profile = src.profile
    transform = src.transform
    crs = src.crs

print(mask.shape)

(1, 262, 262)


In [99]:
np.unique(mask, return_counts=True)

(array([  0, 255], dtype=uint8), array([67927,   717], dtype=int64))

In [100]:
256*256

65536

In [89]:
import numpy as np
np.mean(image, axis=(1, 2))  # Mean pixel value for each band

array([ 312.65522766,  382.08204651,  472.25109863,  245.26615906,
        512.71195984, 1395.00822449, 1770.57844543, 1710.12661743,
       1910.38334656, 1827.64268494,  772.49467468,  375.8956604 ,
          4.93182373])

In [124]:
os.path.isfile(os.path.join("../data/mango_dataset/MF_huggingface/train/images", subset_list[0]))

for i in range(len(subset_list)):
    if os.path.isfile(os.path.join("../data/mango_dataset/MF_huggingface/train/images", subset_list[i])):
        file_name = os.path.join("../data/mango_dataset/MF_huggingface/train/images", subset_list[i])
        
        with rasterio.open(file_name) as src:
            image = src.read()          # Shape: (bands, height, width)
            profile = src.profile
            transform = src.transform
            crs = src.crs

        if image.shape[1] < 256 or image.shape[2] < 256:
            print(image.shape)
        # shutil.copy(os.path.join("../data/mango_dataset/MF_huggingface/train/images", subset_list[i]), os.path.join("../data/mango_dataset/mango_subset/images", subset_list[i]))
        # print(subset_list[i])

(13, 256, 255)
(13, 255, 256)
(13, 255, 255)
(13, 256, 255)
(13, 256, 255)
(13, 256, 255)
(13, 255, 255)
(13, 255, 256)
(13, 256, 255)
(13, 255, 256)
(13, 256, 255)
(13, 255, 255)
(13, 256, 255)
(13, 256, 255)
(13, 256, 255)
(13, 255, 256)
(13, 255, 256)
(13, 255, 255)
(13, 255, 256)


In [44]:
mask_list[[0, 1,4]]

TypeError: list indices must be integers or slices, not list

In [ ]:
"weak_pos", "neg"

In [24]:
img_list[:5]

['region_mid_pos1.tif',
 'region_mid_pos10.tif',
 'region_mid_pos1000.tif',
 'region_mid_pos1001.tif',
 'region_mid_pos1002.tif']